In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU Name: Tesla T4


In [ ]:
#Testing VGG_
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class VGG_(nn.Module):
    def __init__(self):
        super(VGG_, self).__init__()
        self.conv11 = nn.Conv2d(3, 64, 3, padding=1, stride=1)
        self.conv12 = nn.Conv2d(64, 64, 3, padding=1, stride=1)

        self.conv21 = nn.Conv2d(64, 128, 3, padding=1, stride=1)
        self.conv22 = nn.Conv2d(128, 128, 3, padding=1, stride=1)

        self.conv31 = nn.Conv2d(128, 256, 3, padding=1, stride=1)
        self.conv32 = nn.Conv2d(256, 256, 3, padding=1, stride=1)

        self.conv41 = nn.Conv2d(256, 512, 3, padding=1, stride=1)
        self.conv42 = nn.Conv2d(512, 512, 3, padding=1, stride=1)

        self.conv51 = nn.Conv2d(512, 512, 3, padding=1, stride=1)
        self.conv52 = nn.Conv2d(512, 512, 3, padding=1, stride=1)

        # self.fc1 = nn.Linear(25088, 4096) 
        self.fc1 = nn.Linear(512, 4096) #using 32*32 original CIFAR size image to save memory
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, 10)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()

    def forward(self, x):
        x = F.relu(self.conv11(x))
        x = F.relu(self.conv12(x))
        x = self.maxpool(x)

        x = F.relu(self.conv21(x))
        x = F.relu(self.conv22(x))
        x = self.maxpool(x)

        x = F.relu(self.conv31(x))
        x = F.relu(self.conv32(x))
        x = self.maxpool(x)

        x = F.relu(self.conv41(x))
        x = F.relu(self.conv42(x))
        x = self.maxpool(x)

        x = F.relu(self.conv51(x))
        x = F.relu(self.conv52(x))
        x = self.maxpool(x)

        x = self.flatten(x)

        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.5, training=self.training)

        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.fc3(x)

        return x

transform = transforms.Compose([
                        #transforms.Resize((224,224)), #using 32*32 original CIFAR size image to save memory
                            transforms.ToTensor(), 
                                transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])

train_dataset = datasets.CIFAR10(root='./data',
                                 train=True,
                                transform=transform,
                                download=True)

test_dataset = datasets.CIFAR10(root='./data',
                                train=False,
                                transform=transform,
                                download=True)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = VGG_().to(device=device)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9, weight_decay=0.0005)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(74):
    correct = 0
    total = 0
    total_loss = 0
    model.train()

    for images, lables in train_dataloader:
        images, lables = images.to(device), lables.to(device)
        
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, lables)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        y_pred = torch.argmax(output, dim=1)
        correct += (y_pred == lables).sum().item()
        total += lables.size(0)

    acc = correct/total
    print(f"Epoch {epoch+1}, Loss={total_loss/len(train_dataloader):.4f}, Acc={acc:.4f}")

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU Name: Tesla T4


In [ ]:
#Testing MobileNetv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class ConvBlock(nn.Module):
    def __init__(self, first):
        super().__init__()
        if first:
            self.conv0 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=2, padding=1, bias=False),
                                                    nn.BatchNorm2d(32),
                                                    nn.ReLU6(inplace=True),
                                                    )

        else:
            self.conv0 = nn.Sequential(nn.Conv2d(in_channels=320, out_channels=1280, kernel_size=1, stride=1, padding=0, bias=False),
                                                    nn.BatchNorm2d(1280),
                                                    nn.ReLU6(inplace=True),
                                                    )
        
    def forward(self, x):
        x = self.conv0(x)
        return x

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, t):
        super().__init__()

        hidden_dim = in_channels * t
        self.use_residual = (stride == 1 and in_channels == out_channels)

        layers = []
        #PointWise
        if t !=1:
            layers.append(nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=hidden_dim, kernel_size=1, bias=False),
                                        nn.BatchNorm2d(hidden_dim),
                                        nn.ReLU6(inplace=True),
                                        ))

        #DepthWise
        layers.append(nn.Conv2d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=3, stride=stride, padding=1, groups=hidden_dim, bias=False))
        layers.append(nn.BatchNorm2d(hidden_dim))
        layers.append(nn.ReLU6(inplace=True))
   
        #PointWise
        layers.append(nn.Sequential(nn.Conv2d(in_channels=hidden_dim, out_channels=out_channels, kernel_size=1, bias=False),
                                        nn.BatchNorm2d(out_channels),
                                        ))
        
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        else:
            return self.block(x)


class MobileNetv2_(nn.Module):
    '''
    Docstring for MobileNetv2_
    Here t=expansion, c=out_channel, n=num_blocks, s=stride
    '''
    def __init__(self, num_classes=1000):
        super().__init__()
        #224^2 * 3 Conv2d t=-, c=32, n=1, s=2
        self.conv0 = ConvBlock(first=True)

        config = [# t, c, n, s
                        (1, 16, 1, 1),
                        (6, 24, 2, 2),
                        (6, 32, 3, 2),
                        (6, 64, 4, 2),
                        (6, 96, 3, 1),
                        (6, 160, 3, 2),
                        (6, 320, 1, 1),]
        
        #112^2 * 16 bottleneck t=6, c=24, n=2, s=2
        self.features = nn.ModuleList()
        self.in_channels=32

        for t,c,n,s in config:
            for i in range(n):
                stride = s if i == 0 else 1
                self.features.append(InvertedResidual(in_channels=self.in_channels, out_channels=c, stride=stride, t=t))

                self.in_channels = c
        
        self.conv1 = ConvBlock(first=False)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.conv0(x)

        for layer in self.features:
            x = layer(x)

        x = self.conv1(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        x = self.dropout(x)
        x = self.fc(x)

        return x
    
device = ('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

transform = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

traindataset = datasets.CIFAR100(root='./path', train=True, transform=transform, download=True)
testdataset = datasets.CIFAR100(root='./path', train=False, transform=transform, download=True)

traindataloader = DataLoader(dataset=traindataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
testdataloader = DataLoader(dataset=testdataset, batch_size=32, shuffle=False)

model = MobileNetv2_().to(device)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
optimizer = optim.Adam(params=model.parameters(), lr=0.00001)
loss_fn = nn.CrossEntropyLoss()

epoch = 20

for _ in range(epoch):
    total_loss = 0
    loss = 0
    correct = 0
    model.train()

    for images, labels in testdataloader:
        images, labels = images.to(device), labels.to(device)

        output = model(images)

        loss = loss_fn(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        y_pred = torch.softmax(output, dim=1)
        y_pred = torch.argmax(y_pred, dim=1)

acc = (output == labels).sum().item()
print(f"Epoch {epoch+1}, Loss={total_loss/len(traindataloader):.4f}, Acc={acc:.4f}")
        